# Musanga V4 — NB00: 78 exhaustive field plots, UAV crown consistency and modality inventory

**V4 field reference revised 12 November 2026.** Sections 00–05 audit RGB, DSM, SfM LAS and MS within the supplied modelling ROI. Sections **06–08** use Nadine's latest **two GeoPackages**, including all **78** field plots and **3,324** classified crown polygons. No U-Net model training or existing image files are modified.

**Interpretation:** `pct_BA_musanga` is DBH-derived relative stem basal area. It is independent of UAV annotations and a structural-dominance proxy, but not a direct measurement of projected crown area. The plot-wise UAV crown fraction is computed as digitized Musanga polygon union divided by the census plot footprint. Yoko `12Y1` has 0.12 ha inventoried within a 0.16 ha ROI and is excluded from the primary regression pending the location of the surveyed subplots.


## 00 — Environment and read-only source registries

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas numpy matplotlib laspy
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import json, hashlib, re, warnings
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
from IPython.display import display
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT_CANDIDATES=[
    Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
    Path('/content/drive/MyDrive/Musanga/Final'),
]
ROOT=next((p for p in ROOT_CANDIDATES if p.exists()),None)
assert ROOT is not None, f'Could not find Final in {ROOT_CANDIDATES}'
DATA=ROOT/'Data'
assert DATA.exists(),f'Data is missing: {DATA}'
V31=ROOT/'Outputs/V3_1'
V31_MANIFEST=V31/'00_input_audit/00_input_manifest.json'
assert V31_MANIFEST.exists(),f'Approved V3_1 input manifest missing: {V31_MANIFEST}'
base=json.loads(V31_MANIFEST.read_text())
assert base.get('schema_version')=='v3_1_input_audit_kvo_final', base.get('schema_version')
OUT=ROOT/'Outputs/v4/00_input_audit'
OUT.mkdir(parents=True,exist_ok=True)
print('DATA (read-only):',DATA)
print('V3_1 manifest (read-only):',V31_MANIFEST)
print('V4 output:',OUT)
print('V3_1 annotation reference:',base.get('reference_version'))


## 01 — Discover RGB mosaics, DSM rasters and LAS without guessing site mappings

In [ ]:
FORESTS=('Baego','MOS','Yangambi','Yoko')
# Candidates only. Nothing is approved from filename matching alone.
SITE_KEYWORDS={
    'Baego': ('baego',),
    'MOS': ('mos_', 'mos-', 'mos1', 'mos5', 'mos 1', 'mos 5', 'mlt_mos'),
    'Yangambi': ('yangambi','carlier2020','ybjoseph','ybforgeo'),
    'Yoko': ('yoko',),
}
RGB_NAMES=[
    'Baego.tif',
    'Mos_1_2_transparent_mosaic_group1_georef.tif',
    'Mos_3_4_transparent_mosaic_group1_georef.tif',
    'Mos_5_6_transparent_mosaic_group1_georef.tif',
    'Yangambi_georef.tif',
    'Yoko_kvo_georef.tif',
]
LAS_EXPECTED={
    'MOS': ['mlt_Mos_1_2_Green_densified_point_cloud_georef.las',
            'mlt_Mos_3_4_Green_densified_point_cloud_georef.las',
            'mlt_Mos_5_6_Green_densified_point_cloud_georef.las'],
    'Yangambi': ['Carlier2020_Aug20_flights_group1_densified_point_cloud_part_'+str(i)+'_georef.las' for i in range(1,7)]
              + ['YBJoseph_Green_densified_point_cloud_georef.las',
                 'YBForgeo_Green_densified_point_cloud_georef.las'],
    'Yoko': ['Yoko_survey_georef.las'],
    'Baego': [
        'Nadine_Baego_B2mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_B3mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_C2mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_C3mlt_2023_Green_densified_point_cloud.las',
    ],
}
# Bounded to project Data; metadata-only scan, no heavy raster/LAS loads.
paths=[p for p in DATA.rglob('*') if p.is_file() and p.suffix.lower() in ('.tif','.tiff','.las','.laz')]
rows=[]
for p in paths:
    name=p.name.lower()
    tags=[site for site, keys in SITE_KEYWORDS.items() if any(k in name or k in str(p.parent).lower() for k in keys)]
    rows.append({'path':str(p),'name':p.name,'type':'LAS' if p.suffix.lower() in ('.las','.laz') else 'raster',
                 'site_candidate':'|'.join(tags) if tags else '',
                 'dsm_candidate':bool(re.search(r'(^|[_\\-])(dsm|surface|elevation)([_\\-]|$)',p.stem,re.I)),
                 'bytes':p.stat().st_size})
inventory=pd.DataFrame(rows,columns=['path','name','type','site_candidate','dsm_candidate','bytes'])
inventory.to_csv(OUT/'00_file_inventory.csv',index=False)
display(inventory.groupby(['type','site_candidate'],dropna=False).size().rename('n').reset_index())
print('Rasters with DSM-style names:')
display(inventory.query('dsm_candidate').loc[:,['site_candidate','name','path']])
print('Known V3_1 active RGB mosaics:')
for name in RGB_NAMES:
    matches=inventory[inventory.name.str.lower()==name.lower()]
    print(name,'=>',matches.path.tolist())


## 02 — Fast raster metadata audit (no pixel reads)

Read TIFF headers only. Do not downsample massive UAV mosaics over Google Drive in NB00.


In [ ]:
def raster_info(path):
    # FAST: TIFF metadata only. No ds.read(), no resampling, no large Drive I/O.
    with rasterio.open(path) as ds:
        b=ds.bounds
        return {'path':str(path),'name':Path(path).name,
                'crs':str(ds.crs) if ds.crs else None,
                'width':ds.width,'height':ds.height,'bands':ds.count,
                'px_x':ds.transform.a,'px_y':ds.transform.e,
                'left':b.left,'bottom':b.bottom,'right':b.right,'top':b.top,
                'nodata':ds.nodata,'dtype':str(ds.dtypes[0]),
                'block_shapes':str(ds.block_shapes[:2]),
                'sample_valid_fraction':np.nan,
                'sample_z_min':np.nan,'sample_z_p50':np.nan,'sample_z_p99':np.nan}
raster_rows=[]
for p in inventory.query("type == 'raster'").path:
    try:
        raster_rows.append({**raster_info(p),'error':''})
    except Exception as exc:
        raster_rows.append({'path':p,'name':Path(p).name,'error':repr(exc)})
raster_df=pd.DataFrame(raster_rows)
raster_df.to_csv(OUT/'00_raster_metadata.csv',index=False)
display(raster_df[['name','crs','px_x','px_y','bands','width','height','error']])
print('Metadata audit complete. Valid-data fraction intentionally deferred to small local samples in NB01.')


## 03 — ROI-specific RGB–DSM coverage for all **six** mosaics

The denominator is `ROI.shp ∩ RGB mosaic bounds`, **not the full RGB bounding rectangle**. Find DSM candidate rasters anywhere under `Final/Data`, prioritize matching site/mosaic groups and maximum polygon intersection of the intended ROI, and report ambiguous/failed assignments. Report both geometry-only coverage and the need for pixel-validity and co-registration checks. The ROI is the modelling region: no requirement for any DSM or MS to cover the unused image outside the ROI.

In [ ]:
# NB00.03 — ALL six RGB mosaic / DSM coverage on the user's ROI ONLY
import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union
from rasterio.warp import transform_geom
from rasterio.crs import CRS
ROI_PATH=DATA/'Ortho_RGB'/'ROI.shp'
assert ROI_PATH.is_file(),f'Missing modelling ROI: {ROI_PATH}'
ROI_GDF=gpd.read_file(ROI_PATH)
assert ROI_GDF.crs is not None and len(ROI_GDF)>0,'ROI has no CRS / polygons'
ROI_GDF=ROI_GDF.loc[ROI_GDF.geometry.notna() & ~ROI_GDF.geometry.is_empty].copy()
ROI_GDF['geometry']=ROI_GDF.geometry.make_valid()
ROI_GEOM=unary_union(ROI_GDF.geometry)
assert ROI_GEOM.area>0
print('Modelling ROI:',ROI_PATH, 'shapes:',len(ROI_GDF))

def mosaic_key(name):
    v=str(name).casefold().replace(' ','_')
    if 'baego' in v:return 'Baego'
    if 'mos' in v:
        for k in ('1_2','3_4','5_6'):
            if k in v:return 'MOS_'+k
        return 'MOS_unknown'
    if 'yangambi' in v or 'ybjoseph' in v or 'ybforgeo' in v:return 'Yangambi'
    if 'yoko' in v:return 'Yoko'
    return 'unknown'

def raster_bounds_geom_in_crs(path,target_crs):
    with rasterio.open(path) as ds:
        assert ds.crs is not None,f'Raster CRS undefined: {path}'
        if ds.crs==target_crs:
            bounds_geom=box(*ds.bounds)
        else:
            bb=transform_bounds(ds.crs,target_crs,*ds.bounds,densify_pts=21)
            bounds_geom=box(*bb)
        return bounds_geom,ds

def candidate_header(path):
    with rasterio.open(path) as ds:
        return dict(path=str(path),name=path.name,key=mosaic_key(path.name),
                    crs=str(ds.crs),count=ds.count,
                    bounds=(ds.bounds.left,ds.bounds.bottom,ds.bounds.right,ds.bounds.top),
                    resolution=(abs(ds.transform.a),abs(ds.transform.e)),
                    same_grid=False)

# Cached metadata only, no mosaic-sized pixel reads.
RGB_MATCH={}
for name in RGB_NAMES:
    matches=inventory.loc[(inventory.name==name)&
        (inventory.path.str.contains('/Ortho_RGB/',regex=False)), 'path'].tolist()
    assert len(matches)==1,(name,'expected 1 RGB',matches)
    RGB_MATCH[name]=Path(matches[0])
DSM_FILES=sorted(p for p in (DATA/'DSM').rglob('*')
                 if p.is_file() and p.suffix.casefold() in ('.tif','.tiff'))
assert len(DSM_FILES)>=6,'Expected six DSM rasters in Data/DSM'

ROI_RGB={}
for rgb_name,rgb_path in RGB_MATCH.items():
    with rasterio.open(rgb_path) as rgb:
        rgb_crs=rgb.crs
        rgb_bb=box(*rgb.bounds)
        roi_local=ROI_GDF.to_crs(rgb_crs).geometry.union_all()
        support=roi_local.intersection(rgb_bb)
        ROI_RGB[rgb_name]=dict(support=support,crs=rgb_crs,
                               rgb_path=rgb_path,rgb_transform=rgb.transform,
                               roi_area_m2=support.area)
        print(rgb_name, 'ROI ∩ RGB area (ha):',round(support.area/1e4,2))
        assert support.area>0,f'No modelling ROI intersects {rgb_name}'

dsm_candidates=[]
for rgb_name,info in ROI_RGB.items():
    support=info['support']; crs=info['crs']; key=mosaic_key(rgb_name)
    for dsm_path in DSM_FILES:
        with rasterio.open(dsm_path) as dsm:
            if dsm.crs is None: continue
            bb=box(*transform_bounds(dsm.crs,crs,*dsm.bounds,densify_pts=21))
            overlap=support.intersection(bb).area / support.area
            same_grid=(dsm.crs==crs and dsm.transform==info['rgb_transform'])
        k=mosaic_key(dsm_path.name)
        # Spatial support first, but explicit MOS survey ID is mandatory
        # if available to prevent cross-survey mispairing.
        compatible=(k==key or
             (key=='Yangambi' and k=='Yangambi'))
        if compatible:
            dsm_candidates.append(dict(rgb=rgb_name,site_key=key,dsm_name=dsm_path.name,
                dsm_path=str(dsm_path),roi_overlap_fraction=overlap,
                same_grid=same_grid,crs_match=bool(str(dsm.crs)==str(crs))))
dsm_selection=[]
for rgb_name in RGB_NAMES:
    sub=pd.DataFrame([c for c in dsm_candidates if c['rgb']==rgb_name])
    assert len(sub)>0,f'No compatible DSM candidates for {rgb_name}'
    sub=sub.sort_values('roi_overlap_fraction',ascending=False)
    best=sub.iloc[0]
    if len(sub)>1 and sub.iloc[1].roi_overlap_fraction>=best.roi_overlap_fraction-0.01:
        print('DSM AMBIGUOUS within 1% for',rgb_name)
        display(sub[['dsm_name','roi_overlap_fraction']])
    dsm_selection.append(dict(
        rgb=rgb_name,rgb_matches=1,dsm_name=best.dsm_name,dsm_matches=int(len(sub)),
        rgb_path=str(RGB_MATCH[rgb_name]),dsm_path=best.dsm_path,
        crs_match=bool(best.crs_match),same_grid=bool(best.same_grid),
        overlap_fraction_of_rgb_bounds=np.nan, # legacy diagnostic not an acceptance criterion
        roi_overlap_fraction=float(best.roi_overlap_fraction),
        roi_area_ha=ROI_RGB[rgb_name]['roi_area_m2']/1e4,
        extent_qa_pass=bool(best.roi_overlap_fraction>=.95),
        registration_verified=False,
        reason='ROI coverage adequate; pixel validity/registration pending' if best.roi_overlap_fraction>=.95
            else 'ROI-only DSM coverage <95%; inspect valid ROI footprint'))
pair_df=pd.DataFrame(dsm_selection)
pair_df.to_csv(OUT/'00_rgb_dsm_pair_audit.csv',index=False)
pd.DataFrame(dsm_candidates).to_csv(OUT/'00_rgb_dsm_all_candidates.csv',index=False)
display(pair_df[['rgb','dsm_name','roi_area_ha','roi_overlap_fraction',
                 'extent_qa_pass','same_grid','reason']])
print('DSM ROI bbox extent pass:',int(pair_df.extent_qa_pass.sum()),'/',len(pair_df))
print('NOTE: raster bounding-box overlap is NOT valid-pixel overlap or subpixel registration.')


## 04 — SfM LAS header-only audit, including Yangambi's six Carlier parts

In [ ]:
import laspy
las_rows=[]
for p in inventory.query("type == 'LAS'").path:
    try:
        with laspy.open(p) as lf:
            h=lf.header
            las_rows.append({'path':p,'name':Path(p).name,'point_count':int(h.point_count),
                             'crs':str(h.parse_crs()) if h.parse_crs() else '',
                             'xmin':float(h.mins[0]),'ymin':float(h.mins[1]),'zmin':float(h.mins[2]),
                             'xmax':float(h.maxs[0]),'ymax':float(h.maxs[1]),'zmax':float(h.maxs[2]),
                             'point_format':str(h.point_format.id),'version':str(h.version),'error':''})
    except Exception as exc:
        las_rows.append({'path':p,'name':Path(p).name,'error':repr(exc)})
las_df=pd.DataFrame(las_rows)
las_df.to_csv(OUT/'00_las_header_inventory.csv',index=False)
display(las_df)
expected=[(site,name) for site,names in LAS_EXPECTED.items() for name in names]
missing=[(site,name) for site,name in expected if name.lower() not in set(inventory.name.str.lower())]
print('Expected LAS missing from local Data/LAS:',missing)
print('Interpretation: SfM surface reconstruction, not penetrating lidar; do NOT infer DTM, CHM, or foliage profiles.')
print('Carlier parts 1–6 are one logical Yangambi dataset. Do not physically merge until CRS, coordinates and overlap pass.')


## 04B — Multispectral inventory and **ROI-only** coverage, including MOS 3–4

Find all MS rasters under `Final/Data/MS`. Classify each file by survey group and band (or multi-band stack if metadata permits), then measure its extent against the **same ROI ∩ RGB** used by DSM. A file existing in the MS folder is not proof that all four wavelengths are available. Report missing or duplicate bands explicitly. Compare channel coverage on a matched ROI support, independently of full RGB mosaic extents.

In [ ]:
# NB00.04B — ROI-limited MS coverage, including MOS_3_4; no full image reads
MS_DIR=DATA/'MS'
assert MS_DIR.is_dir()
ms_files=sorted(p for p in MS_DIR.rglob('*') if p.is_file()
                and p.suffix.casefold() in ('.tif','.tiff'))
def ms_group(p):
    k=mosaic_key(p.name)
    if k in ('MOS_1_2','MOS_3_4','MOS_5_6'):return k+'_MS'
    if k=='Baego':return 'Baego_MS'
    n=p.name.casefold()
    if 'ybjoseph' in n:return 'Yangambi_YBJoseph_right_MS'
    if 'ybforgeo' in n:return 'Yangambi_YBForgeo_right_MS'
    if k=='Yoko':return 'Yoko_MS'
    return 'UNASSIGNED'
def ms_band_hint(p):
    n=p.stem.casefold().replace('-','_')
    if 'red edge' in n or 'red_edge' in n or 'rededge' in n:return 'red_edge'
    for band in ('green','nir','red'):
        if re.search(r'(^|[_ ])'+band+r'([_ ]|$)',n):return band
    return 'multiband_or_unknown'
ms_meta=[]
for p in ms_files:
    try:
        with rasterio.open(p) as ds:
            b=ds.bounds
            ms_meta.append(dict(group=ms_group(p),band_hint=ms_band_hint(p),
                name=p.name,path=str(p),bands=ds.count,
                band_descriptions=str(ds.descriptions),crs=str(ds.crs) if ds.crs else '',
                width=ds.width,height=ds.height,
                pixel_x_m=float(ds.transform.a),pixel_y_m=float(ds.transform.e),
                left=b.left,bottom=b.bottom,right=b.right,top=b.top,
                nodata=ds.nodata,dtype=str(ds.dtypes[0]),error=''))
    except Exception as e:
        ms_meta.append(dict(group=ms_group(p),band_hint=ms_band_hint(p),
                            name=p.name,path=str(p),error=repr(e)))
MS_META=pd.DataFrame(ms_meta)
MS_META.to_csv(OUT/'00_ms_raster_headers.csv',index=False)
display(MS_META.groupby(['group','band_hint'],dropna=False).size().rename('files').reset_index())
# Legacy MS group names retained for downstream NB01 compatibility.
RGB_REFERENCE={
    'Baego_MS':'Baego.tif',
    'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
    'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
    'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
    'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
    'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif',
    'Yoko_MS':'Yoko_kvo_georef.tif'
}
paired=[]
for group,rgb_name in RGB_REFERENCE.items():
    support=ROI_RGB[rgb_name]['support']
    crs=ROI_RGB[rgb_name]['crs']
    for _,rec in MS_META[MS_META.group.eq(group)].iterrows():
        if rec.get('error','') or not rec.get('crs'):continue
        bb=box(*transform_bounds(rec['crs'],crs,
            rec.left,rec.bottom,rec.right,rec.top,densify_pts=21))
        fraction=support.intersection(bb).area/max(support.area,1e-12)
        paired.append(dict(group=group,band_hint=rec.band_hint,name=rec['name'],
                           path=rec.path,rgb_reference=rgb_name,
                           roi_overlap_fraction=float(fraction),
                           bbox_fraction_of_rgb=np.nan,
                           rgb_crs=str(crs),ms_crs=rec.crs,
                           same_crs=bool(str(crs)==rec.crs),
                           bands=int(rec.bands),
                           acquisition_match_verified=False,
                           roi_valid_overlap_verified=False))
MS_OVERLAP=pd.DataFrame(paired)
MS_OVERLAP.to_csv(OUT/'00_ms_rgb_extent_audit.csv',index=False)
if len(MS_OVERLAP):
    display(MS_OVERLAP[['group','band_hint','name','bands','roi_overlap_fraction']])
else:
    print('No matched multispectral files located.')
print('ROI coverage means bounding-box coverage ONLY, not valid MS pixels.')
# Report 4-band availability without inventing channels for group1 stacks.
requirements=['red','green','red_edge','nir']
summaries=[]
for group,rgb_name in RGB_REFERENCE.items():
    part=MS_OVERLAP[MS_OVERLAP.group.eq(group)] if len(MS_OVERLAP) else pd.DataFrame()
    coverage={}
    for band in requirements:
        cand=part[part.band_hint.eq(band)] if len(part) else pd.DataFrame()
        coverage[band]=float(cand.roi_overlap_fraction.max()) if len(cand) else np.nan
    multiband=part[(part.band_hint=='multiband_or_unknown') & (part.bands>=4)] if len(part) else pd.DataFrame()
    summaries.append(dict(group=group,rgb=rgb_name,**coverage,
         min_explicit_band_coverage=np.nanmin(list(coverage.values()))
          if all(np.isfinite(list(coverage.values()))) else np.nan,
         multiband_4plus_candidate_files=int(len(multiband)),
         complete_explicit_4band_roi_bbox=all(np.isfinite(list(coverage.values()))) and
                                      min(coverage.values())>=.95))
MS_SUMMARY=pd.DataFrame(summaries)
display(MS_SUMMARY.round(3))
MS_SUMMARY.to_csv(OUT/'00_ms_roi_4band_summary.csv',index=False)
ms_registry=dict(schema='musanga_v4_ms_inventory_roi_2',
    ms_dir=str(MS_DIR),
    groups={group:MS_META.loc[MS_META.group.eq(group),
                    ['name','path','band_hint','bands']].to_dict('records')
            for group in sorted(MS_META.group.unique())},
    rgb_reference=RGB_REFERENCE,
    roi_support='Ortho_RGB/ROI.shp intersection individual RGB footprint',
    notes=['No full RGB bbox completeness requirement','No Yoko MS presumed absent until audited',
           'Multi-band stack band identities need validation',
           'Pixel validity/radiometry/co-registration must be checked before model comparison'])
(OUT/'00_ms_registry.json').write_text(json.dumps(ms_registry,indent=2))
print('MS coverage audited for all six mosaic references, including MOS 3–4.')


## 05 — Freeze proposed V4 audit contract (no modelling yet)

In [ ]:
audit={
    'schema_version':'v4_nb00_rgb_dsm_sfm_inventory_1',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'inherited_reference_manifest':str(V31_MANIFEST),
    'inherited_reference_schema':base.get('schema_version'),
    'inherited_reference_version':base.get('reference_version'),
    'v3_1_folds_policy':'Reuse frozen 30-circle within-site spatial folds; do not regenerate',
    'v3_1_excluded_original_circle_ids':[6,9,24,25],
    'v3_1_training_exclusion_buffer_m':15,
    'primary_modalities':['RGB','DSM-derived local geometry'],
    'roi_geometry_file':str(ROI_PATH),
    'roi_coverage_denominator':'ROI.shp intersection RGB footprint, never full RGB rectangle',
    'rgb_mosaic_count':int(len(RGB_NAMES)),
    'dsm_roi_coverage_report':str(OUT/'00_rgb_dsm_pair_audit.csv'),
    'ms_roi_coverage_report':str(OUT/'00_ms_roi_4band_summary.csv'),
    'optional_modalities':['MS bands/indices on verified survey overlaps','LAS SfM 3D diagnostics'],
    'ms_inventory_manifest':str(OUT/'00_ms_registry.json'),
    'ms_group_count':int(MS_META.group.nunique()),
    'ms_file_count':int(len(MS_META)),
    'ms_partial_coverage_policy':'Do not extrapolate YBJoseph/YBForgeo MS to Yangambi Carlier left; no Yoko MS',
    'comparison_support_policy':'Matched valid ROI∩RGB∩modality footprint; same frozen folds per comparison',
    'experimental_las':'small stratified crown sample, only if added information beyond DSM',
    'prohibited_unjustified_features':['CHM','DTM','understory profile from SfM'],
    'output_root':str(ROOT/'Outputs/v4'),
    'n_indexed_rasters':int((inventory.type=='raster').sum()),
    'n_indexed_las':int((inventory.type=='LAS').sum()),
    'rgb_dsm_pairings_confirmed':bool(len(pair_df)==len(RGB_NAMES) and pair_df.crs_match.all()),
    'rgb_dsm_extent_qa_pass':bool(pair_df.extent_qa_pass.all()),
    'rgb_dsm_registration_verified':False,
    'next_step':'NB01 after ROI-limited RGB/DSM/MS valid-pixel and registration QA',
}
manifest=OUT/'00_v4_input_audit_manifest.json'
manifest.write_text(json.dumps(audit,indent=2))
print('WROTE:',manifest)
print('RGB–DSM filenames/CRS confirmed:',audit['rgb_dsm_pairings_confirmed'])
print('RGB–DSM extent QA:',audit['rgb_dsm_extent_qa_pass'])
print('Subpixel registration still pending:',not audit['rgb_dsm_registration_verified'])
print('V3_1 remains unchanged:',V31_MANIFEST)
if not audit['rgb_dsm_pairings_confirmed']:
    print('ACTION REQUIRED: investigate unmatched filenames/CRS; check 00_rgb_dsm_pair_audit.csv.')


## 06 — Revised authoritative 78-plot field reference (two GeoPackages)

Inputs in `Final/Data/Field`: **`Plot_Data_BA_nobuffer_ROI 4.gpkg`** (78 plots) and **`Tree_Annotation_georef_nadine_new 1.gpkg`** (3,324 crown polygons). All `annotated=True` plots are exhaustive. A crown with `Class_ID = 1` is Musanga; all other finite classes are non-Musanga. Empty crown metadata fields are not used: polygon-to-plot assignment is spatial. Repair invalid geometries in memory with an explicit 5% area-change quarantine threshold; do not overwrite either source file. Preserve mapped plot geometries, verify their nominal area, and flag the partially inventoried `12Y1`.


In [ ]:
# NB00.06 — Authoritative V4 field78 plot/crown data QA (read-only inputs)
import geopandas as gpd
from shapely import make_valid
from shapely.ops import unary_union
from shapely.geometry import GeometryCollection
FIELD_DIR=DATA/'Field'
PLOT_V4=FIELD_DIR/'Plot_Data_BA_nobuffer_ROI 4.gpkg'
CROWN_V4=FIELD_DIR/'Tree_Annotation_georef_nadine_new 1.gpkg'
for p in (PLOT_V4,CROWN_V4):assert p.is_file(),f'Missing new V4 field input: {p}'
P=gpd.read_file(PLOT_V4).to_crs('EPSG:32635')
C=gpd.read_file(CROWN_V4).to_crs('EPSG:32635')
required={'Plot_name','Site','surface_ha','pct_BA_musanga','annotated'}
assert required.issubset(P.columns),f'Missing plot fields: {required-set(P.columns)}'
assert 'Class_ID' in C.columns
assert len(P)==78,f'Expected 78 plots; found {len(P)}'
assert P.Plot_name.is_unique,'Duplicate field plot names'
assert P.geometry.notna().all() and P.geometry.is_valid.all()
assert P['annotated'].fillna(False).astype(bool).all(),'Not all plots are confirmed exhaustive'
assert pd.to_numeric(P.pct_BA_musanga,errors='coerce').between(0,100).all()
P['mapped_area_ha']=P.geometry.area/10000
P['census_area_ha']=pd.to_numeric(P.surface_ha,errors='coerce')
assert P.census_area_ha.gt(0).all()
P['area_ratio_mapped_to_census']=P.mapped_area_ha/P.census_area_ha
P['area_mismatch']=~np.isclose(P.mapped_area_ha,P.census_area_ha,atol=.003,rtol=.02)
P['validation_geom_source']='supplied_plot_roi'
P['primary_BA_validation']=~((P.Site=='Yoko')&(P.Plot_name.str.upper()=='12Y1'))
assert len(P.loc[~P.primary_BA_validation])==1
# Do not manufacture a centred square for Yoko 12Y1: the 3 observed quadrants
# are not spatially identified. Preserve its supplied polygon as sensitivity only.
assert P.loc[P.Site.eq('Mos'),'census_area_ha'].eq(.25).all(),'MOS census plot must be 0.25 ha'
assert np.isclose(P.loc[P.Site.eq('Mos'),'mapped_area_ha'],.25,rtol=.02).all(),'MOS ROI must be 0.25 ha'
assert len(P.loc[P.Site.eq('Mos')])==20
print('Field reference:',len(P),'plots; all annotated:',P.annotated.all())
display(P.groupby(['Site','area_mismatch']).size().rename('n').reset_index())
display(P.loc[P.area_mismatch,['Plot_name','Site','mapped_area_ha','census_area_ha']].round(3))
if P.area_mismatch.sum()!=1:
    print('WARNING: expected only Yoko 12Y1 to have a census/ROI area mismatch')
# Fix invalid crown polygons without changing the source GeoPackage.
assert pd.to_numeric(C.Class_ID,errors='coerce').notna().all(),'Unknown crown classes'
C['musanga_label']=pd.to_numeric(C.Class_ID,errors='raise').eq(1).astype('uint8')
repaired=[];kept=[];qa=[]
for i,g in enumerate(C.geometry):
    if g is None or g.is_empty:
        qa.append((i,'empty',np.nan));continue
    try:
        if g.is_valid and g.geom_type in ('Polygon','MultiPolygon'):
            fixed=g
        else:
            candidate=make_valid(g)
            if candidate.geom_type in ('Polygon','MultiPolygon'):
                fixed=candidate
            else:
                polys=[x for x in getattr(candidate,'geoms',[])
                        if x.geom_type in ('Polygon','MultiPolygon')]
                fixed=unary_union(polys) if polys else None
        if fixed is None or fixed.is_empty or not fixed.is_valid:
            qa.append((i,'quarantined',np.nan));continue
        change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-9)
        if change>.05:
            qa.append((i,'quarantined_area_change_gt5pct',change));continue
        repaired.append(fixed);kept.append(i)
        qa.append((i,'ok' if g.is_valid else 'repaired',change))
    except Exception as exc:
        qa.append((i,'quarantined_exception',np.nan))
REPAIR_QA=pd.DataFrame(qa,columns=['source_row','status','relative_area_change'])
REPAIR_QA.to_csv(OUT/'00_v4_field_crown_repairs.csv',index=False)
C=C.iloc[kept].copy().reset_index(drop=True)
C.geometry=repaired
assert C.geometry.is_valid.all()
assert len(C)>0
print('Crowns retained:',len(C),'of 3324; Musanga:',int(C.musanga_label.sum()))
display(REPAIR_QA.status.value_counts().rename_axis('status').reset_index(name='n'))
display(C.Class_ID.value_counts().rename_axis('Class_ID').reset_index(name='n'))
if len(C)!=3324:print('REVIEW: some crown geometries were quarantined; inspect repair CSV')
P.drop(columns=['geometry'],errors='ignore').to_csv(OUT/'00_v4_field_plot_attribute_audit.csv',index=False)
P.to_file(OUT/'00_v4_field_plot_support.gpkg',driver='GPKG')
print('Read-only source files preserved; audited 78-plot support exported.')


## 07 — Independent field BA versus Nadine's UAV crown fractions (78 plots)

For each mapped plot, intersect Musanga and non-Musanga crowns **spatially**, clip them to its boundary, then calculate union areas to prevent double-counting. Report both Musanga fraction of the **plot area** (the primary fractional-cover variable) and of **all digitized crowns** (secondary canopy-dominance measure), plus digitized crown coverage and mixed-class intersection area. Report complete 78-plot descriptive results; the **primary regression excludes Yoko 12Y1** because its partial 0.12 ha census cannot be located inside the 0.16 ha polygon. Also compute the 78-plot sensitivity, site-wise metrics, and GIS diagnostics. The number of digitized crowns outside a plot is not by itself evidence of incomplete within-plot annotations.


In [ ]:
# NB00.07 — Robust plot/crown overlay and paired ecological validation
import matplotlib.pyplot as plt
from shapely.strtree import STRtree
from shapely import union_all, set_precision
from scipy.stats import linregress, spearmanr
cc=list(C.geometry);cl=C.musanga_label.to_numpy(int)
tree=STRtree(cc)
def robust_union(geoms,plot_id):
    geoms=[g for g in geoms if g is not None and not g.is_empty and g.area>0]
    if not geoms:return GeometryCollection()
    try:
        a=unary_union(geoms)
        return make_valid(a) if not a.is_valid else a
    except Exception as first:
        try:return make_valid(union_all(geoms,grid_size=.01))
        except Exception as second:
            raise RuntimeError(f'GEOS union failed at {plot_id}: {second}') from first

def safe_clip(crown,plot,plot_id):
    try:return crown.intersection(plot)
    except Exception:
        try:return set_precision(make_valid(crown),.01).intersection(
                       set_precision(make_valid(plot),.01))
        except Exception as exc:raise RuntimeError(
            f'Cannot intersect crown with plot {plot_id}: {exc}') from exc

records=[];boundary_rows=[]
for row in P.itertuples(index=False):
    g=row.geometry
    candidates=tree.query(g,predicate='intersects')
    pos=[];neg=[];crossing=0;outside=0
    for j in candidates:
        k=int(j);part=safe_clip(cc[k],g,row.Plot_name)
        if part.is_empty or part.area<=0:continue
        if not g.covers(cc[k]):crossing+=1
        (pos if cl[k]==1 else neg).append(part)
    up=robust_union(pos,row.Plot_name)
    un=robust_union(neg,row.Plot_name)
    both=robust_union([up,un],row.Plot_name)
    ap=float(up.area);an=float(un.area);au=float(both.area);area=float(g.area)
    assert area>0
    overlap=max(0,ap+an-au)
    records.append(dict(
        Plot_name=str(row.Plot_name),Site=str(row.Site),source=str(row.source),
        field_BA_pct=float(row.pct_BA_musanga),
        musanga_crown_area_m2=ap,non_musanga_crown_area_m2=an,
        digitized_union_area_m2=au,overlapping_label_area_m2=overlap,
        plot_area_m2=area,mapped_area_ha=float(row.mapped_area_ha),
        census_area_ha=float(row.census_area_ha),
        crown_fraction_plot_pct=100*ap/area,
        musanga_fraction_digitized_crowns_pct=100*ap/au if au>0 else np.nan,
        digitized_crown_coverage_pct=100*au/area,
        n_musanga_polygons=len(pos),n_non_musanga_polygons=len(neg),
        n_boundary_crossing_polygons=crossing,
        geometry_source='supplied_plot_roi',
        primary_BA_validation=bool(row.primary_BA_validation)))
FIELD_COMPARE=pd.DataFrame(records)
assert len(FIELD_COMPARE)==78 and FIELD_COMPARE.Plot_name.is_unique
FIELD_COMPARE.to_csv(OUT/'00_v4_field_BA_vs_Nadine_78plots.csv',index=False)
# Keep old artifact name distinct, don't silently overwrite results from 65 plots.
print('Overlay records:',len(FIELD_COMPARE),
      '| plots with >0.1 m2 class overlap:',int((FIELD_COMPARE.overlapping_label_area_m2>.1).sum()),
      '| plots without any digitized crown:',int(FIELD_COMPARE.digitized_union_area_m2.eq(0).sum()))
display(FIELD_COMPARE.groupby('Site')[
    ['field_BA_pct','crown_fraction_plot_pct',
     'musanga_fraction_digitized_crowns_pct','digitized_crown_coverage_pct']
].agg(['size','mean','median']).round(2))

def compare_stats(frame, metric,site,cohort):
    z=frame[['field_BA_pct',metric]].replace([np.inf,-np.inf],np.nan).dropna()
    x=z.field_BA_pct.to_numpy(float);y=z[metric].to_numpy(float)
    q=dict(cohort=cohort,site=site,metric=metric,n=len(z),
           bias_pp=float(np.mean(y-x)) if len(x) else np.nan,
           mae_pp=float(np.mean(abs(y-x))) if len(x) else np.nan)
    if len(x)>2 and np.std(x)>0 and np.std(y)>0:
        r=linregress(x,y)
        q.update(Pearson_r=r.rvalue,Pearson_p=r.pvalue,
                 Spearman_rho=spearmanr(x,y).statistic,
                 Regression_R2=r.rvalue**2,slope=r.slope,intercept=r.intercept)
    return q

stats=[]
for cohort,frame in [('PRIMARY_excludes_partial_Yoko_12Y1',
                       FIELD_COMPARE[FIELD_COMPARE.primary_BA_validation]),
                      ('SENSITIVITY_all78',FIELD_COMPARE)]:
    for site,group in [('ALL',frame),*frame.groupby('Site')]:
        for metric in ['crown_fraction_plot_pct','musanga_fraction_digitized_crowns_pct']:
            stats.append(compare_stats(group,metric,site,cohort))
FIELD_STATS=pd.DataFrame(stats)
FIELD_STATS.to_csv(OUT/'00_v4_field_BA_vs_Nadine_78plot_statistics.csv',index=False)
display(FIELD_STATS.round(3))
fig,axes=plt.subplots(1,2,figsize=(13,5.4),sharex=True,layout='constrained')
primary=FIELD_COMPARE[FIELD_COMPARE.primary_BA_validation]
for ax,metric,title in zip(axes,
        ['crown_fraction_plot_pct','musanga_fraction_digitized_crowns_pct'],
        ['Musanga crown / plot area','Musanga crown / digitized crown area']):
    for site,grp in primary.groupby('Site'):
        ax.scatter(grp.field_BA_pct,grp[metric],s=45,alpha=.8,label=f'{site} (n={len(grp)})')
    omitted=FIELD_COMPARE[~FIELD_COMPARE.primary_BA_validation]
    ax.scatter(omitted.field_BA_pct,omitted[metric],marker='x',
               s=85,color='black',label='Yoko 12Y1 (sensitivity)')
    ax.plot([0,100],[0,100],'--',color='grey',lw=1)
    subset=primary[['field_BA_pct',metric]].dropna()
    if len(subset)>2 and subset.field_BA_pct.std()>0:
        r=linregress(subset.field_BA_pct,subset[metric])
        xx=np.array([0,100]);ax.plot(xx,r.intercept+r.slope*xx,color='black',lw=1.5)
        ax.text(.03,.97,f'primary n={len(subset)}; r={r.rvalue:.3f}; slope={r.slope:.3f}',
                va='top',transform=ax.transAxes,fontsize=9)
    ax.set(xlim=(0,100),ylim=(0,100),title=title,
           xlabel='Field Musanga basal area (%)',ylabel='Nadine UAV Musanga (%)')
    ax.grid(alpha=.2)
axes[1].legend(fontsize=8,loc='lower right')
fig.suptitle('Independent field structural dominance vs exhaustive UAV crown delineations')
fig.savefig(OUT/'00_v4_field_BA_vs_Nadine_78plots.png',dpi=220,bbox_inches='tight')
fig.savefig(OUT/'00_v4_field_BA_vs_Nadine_78plots.pdf',bbox_inches='tight')
plt.show()
# ArcGIS diagnostic with plot-level values
P.merge(FIELD_COMPARE.drop(columns=['Site','source']),on='Plot_name',
        how='left',validate='one_to_one').to_file(
            OUT/'00_v4_field_plot_validation.gpkg',driver='GPKG')
print('Saved 78-plot comparison, paired statistics and ArcGIS diagnostics:',OUT)


## 08 — Frozen V4 field78 contract and controlled modality experiments

The updated reference contains 78 exhaustive field plots and 3,324 crown objects. Preserve earlier V3_1 circle and site annotation manifests as separate sources. NB01 must be revised to consume this **field78 schema**; its older `field65` paths and strict 65-row assertions are obsolete. For fair RGB/DSM/MS/SfM LAS comparisons, compare on common valid ROI support and identical spatial holdout folds. Full training is a later notebook.


In [ ]:
# NB00.08 — Machine-readable updated V4 field78 contract
C31=Path(base['exhaustive_annotations']['positive_polygons'])
R31=Path(base['exhaustive_annotations']['complete_roi_boundaries'])
assert C31.name=='Circles_Annotations_kvo_final.shp'
assert R31.name=='Circles_ROI_NAdine.shp'
contract={
  'schema':'musanga_v4_field78_input_audit_2026_11',
  'plot_file':str(PLOT_V4),
  'plot_layer':'Plot_Data_BA_nobuffer_ROI',
  'field_fraction_column':'pct_BA_musanga',
  'n_field_plots':int(len(P)),
  'n_primary_field_plots':int(P.primary_BA_validation.sum()),
  'exhaustive_plot_crowns':str(CROWN_V4),
  'n_raw_plot_crowns':3324,
  'n_valid_plot_crowns':int(len(C)),
  'crown_musanga_class_id':1,
  'other_class_ids':'negative_crown_objects',
  'exhaustive_plot_support':'supplied validated field plot polygons; Yoko 12Y1 partial-census flagged',
  'field_support_qa':str(OUT/'00_v4_field_plot_support.gpkg'),
  'field_BA_comparison':str(OUT/'00_v4_field_BA_vs_Nadine_78plots.csv'),
  'field_BA_stats':str(OUT/'00_v4_field_BA_vs_Nadine_78plot_statistics.csv'),
  'field_validation_gpkg':str(OUT/'00_v4_field_plot_validation.gpkg'),
  'circle_positive_crowns':str(C31),
  'circle_complete_roi':str(R31),
  'partial_site_crowns':base['shared_annotations'],
  'ablation_modalities':['RGB','RGB_DSM','RGB_MS','RGB_LAS_features','RGB_DSM_MS','RGB_DSM_MS_LAS'],
  'modality_policy':'Compare paired models on identical valid ROI pixels and the same frozen spatial folds',
  'splits_policy':'Connected plot–circle overlaps must be held out together; buffer held-out support',
  'geometry_exception_policy':'Yoko 12Y1 uses 0.16-ha supplied ROI but only 0.12-ha partial census; exclude primary BA comparison',
  'data_root':str(DATA)
}
dest=OUT/'00_v4_field78_contract.json'
dest.write_text(json.dumps(contract,indent=2))
display(pd.DataFrame([{'component':k,'value':str(v)} for k,v in contract.items()
    if k in ('plot_file','n_field_plots','n_primary_field_plots',
             'field_fraction_column','exhaustive_plot_crowns',
             'circle_positive_crowns')]))
print('V4 updated 78-plot contract ready:',dest)
print('IMPORTANT: NB01 still needs migration from field65 to field78 before use.')
